# H5. VISION Track 2 — 합성 데이터로 표면 결함 탐지 고도화 (DIAG)

### 개요

실제 학술대회 과제를 **그대로 재현**해 보는 실습입니다. CVPR/ECCV/ICCV의 **VISION Workshop Challenge — Track 2 (Data Generation for Defect Detection)** 의 과제 정의를 틀로 삼습니다.

> **Track 2 과제**: *제한된 라벨 데이터만 주어진 상황에서, 합성 데이터를 생성해 결함 탐지 모델의 성능(AP)을 끌어올려라.*

이 틀 안을 공개 코드인 **DIAG** (CBMI 2024, `intelligolabs/DIAG`) 로 채웁니다. DIAG는 **학습 불필요(zero-shot)** 로 결함을 생성하는 파이프라인입니다.

| 항목 | 내용 |
|------|------|
| **대회 과제** | VISION Track 2 — 합성으로 탐지 성능(AP) 향상 |
| **데이터셋** | KSDD2 (Kolektor Surface-Defect Dataset 2) — 제조 표면 결함 |
| **생성 방법** | DIAG — SDXL inpainting에 **텍스트 + 영역(마스크)** 지정 (학습 불필요) |
| **탐지 모델** | ResNet-50 (이진 분류: 정상 vs 결함) |
| **평가 지표** | AP (Average Precision) |

**파이프라인**
1. DIAG 공식 코드 클론 & Colab용 의존성 설치
2. KSDD2(제조 표면결함) 다운로드 & 전처리
3. DIAG의 합성 전략 이해 (텍스트 + 영역 지정)
4. SDXL로 합성 결함 이미지 생성
5. **실험**: 합성 데이터 없이(Baseline) vs 있을 때 ResNet-50의 AP 비교

> **실행 환경 — 먼저 읽어 주세요**
>
> - 상단 **런타임 → 런타임 유형 변경 → GPU(T4)** 로 설정하세요. GPU가 없으면 생성이 사실상 불가능합니다.
> - DIAG는 **SDXL(약 7GB 다운로드)** 을 사용합니다. T4에서 돌아가지만 **생성이 느립니다**. 그래서 이 실습은 생성량을 줄인 **데모 설정**으로 구성했습니다.
> - 이 노트북은 DIAG 공식 코드를 Colab에 맞춰 옮긴 것이라, 데이터 다운로드·라이브러리 호환에서 **약간의 조정이 필요할 수 있습니다**. 막히면 각 셀의 안내를 참고하세요.


### 실행 환경 점검

먼저 GPU가 정상적으로 잡혔는지 확인합니다. <span style="color:#E74C3C; font-weight:bold">Tesla T4</span> 정도면 충분합니다(생성은 느릴 수 있음).

In [ ]:
!nvidia-smi -L
import torch
print("CUDA 사용 가능:", torch.cuda.is_available())
print("PyTorch:", torch.__version__)

In [ ]:
import os
import sys
import re
import glob
import time
import logging
import subprocess

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 책의 다른 실습과 동일한 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)s — %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("VISION-Track2")


class Timer:
    """단계별 소요 시간을 측정하는 컨텍스트 매니저."""
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        self.t0 = time.time()
        log.info(f"[START] {self.name}")
        return self
    def __exit__(self, *args):
        log.info(f"[DONE] {self.name} — {time.time() - self.t0:.1f}s")

### 1. DIAG 공식 코드 가져오기 & 의존성 설치

DIAG 저장소를 클론합니다. 핵심 스크립트는 세 개입니다.

- `ksdd2_preprocess.py` — KSDD2를 학습/평가용으로 정리
- `generate_augmented_images.py` — SDXL로 합성 결함 이미지 생성
- `train_ResNet50.py` — (합성을 더한) 데이터로 ResNet-50 학습·평가

> **왜 requirements.txt를 그대로 안 쓰나**
>
> DIAG의 `requirements.txt`는 <span style="color:#2980B9">옛 버전 torch(1.12+cu113)</span>에 고정돼 있어, Colab 기본 torch와 충돌합니다.
> 그래서 torch는 **Colab 기본 버전을 그대로 쓰고**, 필요한 라이브러리(diffusers 등)만 따로 설치합니다.

In [ ]:
with Timer("DIAG 클론 & 의존성 설치"):
    if not os.path.exists("/content/DIAG"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/intelligolabs/DIAG.git",
                        "/content/DIAG"], check=True)
    # torch는 Colab 기본 버전 사용. 생성/학습에 필요한 패키지만 설치.
    !pip install -q "diffusers>=0.27" transformers accelerate safetensors \
                    scikit-learn pandas matplotlib tqdm
    log.info("  설치 완료")

os.chdir("/content/DIAG")
print("작업 디렉터리:", os.getcwd())

### 2. 데이터셋 준비 — KSDD2 (제조 표면 결함)

**KSDD2 (Kolektor Surface-Defect Dataset 2)** 는 전기 정류자(commutator) 표면을 촬영한 실제 제조 검사 데이터셋입니다. 정상 이미지는 많고 결함 이미지는 적은 전형적인 **불균형** 구조라, "합성으로 소수 클래스를 보강한다"는 우리 과제에 딱 맞습니다.

> **데이터 다운로드**
>
> KSDD2는 <span style="color:#E74C3C; font-weight:bold">vicos 공식 사이트</span>에서 받아야 합니다(약 1~2GB).
> 아래 셀은 공개 링크로 내려받기를 시도하지만, 막히면 [공식 페이지](https://www.vicos.si/resources/kolektorsdd2/)에서
> 직접 받아 `/content/ksdd2`에 풀어 주세요. 폴더 안에 `train/`, `test/` 가 보이면 됩니다.

In [ ]:
with Timer("KSDD2 다운로드 & 압축 해제"):
    os.makedirs("/content/ksdd2", exist_ok=True)
    # vicos 공식 배포 링크 (실패 시 위 안내대로 수동 다운로드)
    if not glob.glob("/content/ksdd2/**/*.png", recursive=True):
        !wget -q --show-progress -O /content/ksdd2.zip https://go.vicos.si/kolektorsdd2 || echo "자동 다운로드 실패 → 수동 다운로드 필요"
        !unzip -q -o /content/ksdd2.zip -d /content/ksdd2 || echo "압축 해제 실패 (수동 업로드 후 재실행)"
    n_png = len(glob.glob("/content/ksdd2/**/*.png", recursive=True))
    log.info(f"  KSDD2 이미지 수: {n_png}")
    assert n_png > 0, "KSDD2가 준비되지 않았습니다. 위 안내대로 수동 다운로드 후 다시 실행하세요."

with Timer("KSDD2 전처리 (DIAG)"):
    # DIAG의 전처리: 평가/증강에 쓸 정리된 사본 생성
    subprocess.run([sys.executable, "ksdd2_preprocess.py",
                    "--src_dir", "/content/ksdd2",
                    "--dst_dir", "/content/ksdd2_pre"], check=True)
    log.info("  전처리 완료 → /content/ksdd2_pre")

### 3. DIAG의 합성 전략 — 텍스트 + 영역 지정 (학습 불필요)

DIAG의 아이디어는 우리가 앞 세션에서 본 **"텍스트(무엇) + 마스크(어디)"** 와 정확히 같습니다. 차이는 **생성 모델을 따로 학습시키지 않는다**는 점입니다.

1. 정상 이미지를 한 장 가져온다.
2. 결함을 넣을 **영역(마스크)** 을 지정한다 — "여기에"
3. **텍스트 프롬프트**로 결함 종류를 묘사한다 — "이런 결함을"
4. 사전학습된 **SDXL inpainting** 이 그 영역에 결함을 그려 넣는다.

> **핵심 — "in-distribution" 생성**
>
> DIAG는 정상 이미지 위 지정한 영역에만 결함을 채우므로, 생성물이 <span style="color:#27AE60; font-weight:bold">원래 제품 분포 안(in-distribution)</span>에 머뭅니다.
> 배경이 통째로 바뀌지 않아 "현실에 없는 결함"이 될 위험이 줄고, 탐지기 학습에 바로 도움이 됩니다.

실제로 어떤 프롬프트를 쓰는지 DIAG 코드에서 직접 확인해 봅니다 — 원본 코드를 직접 읽어 보는 단계입니다.

In [ ]:
# DIAG의 생성 스크립트에서 프롬프트/핵심 로직 일부를 직접 확인
with open("/content/DIAG/generate_augmented_images.py") as f:
    src = f.read()

# 프롬프트가 정의된 줄 주변만 발췌해서 출력
for i, line in enumerate(src.splitlines()):
    if "prompt" in line.lower():
        print(f"{i:4d}: {line}")

### 4. 합성 결함 이미지 생성 (SDXL inpainting)

이제 합성 결함 이미지를 만듭니다. **이 단계가 노트북에서 가장 오래 걸립니다**(다운로드보다 생성 자체가 느립니다).

> **⏱️ 예상 시간 — 가장 무거운 단계**
>
> - **첫 실행 SDXL 준비**: 모델 가중치 <span style="color:#E74C3C; font-weight:bold">약 7GB 다운로드 + 로드</span>로 약 **3~5분**(세션당 1회).
> - **이미지 1장 생성**: T4에서 대략 **20~40초**(인페인팅 50스텝 기준).
> - **총 생성 시간**: `IMGS_PER_PROMPT × 프롬프트 수` 만큼 만듭니다. 기본값 `IMGS_PER_PROMPT=10`이면 대략 **10~20분**(첫 실행은 위 다운로드가 추가).
> - 시간을 줄이려면 `IMGS_PER_PROMPT`를 낮추세요(예: 3). 품질·다양성을 높이려면 키우되 시간이 비례해서 늘어납니다.

In [ ]:
# --- 생성 설정 ---
IMGS_PER_PROMPT = 3     # 프롬프트당 생성 이미지 수 (데모용으로 작게)
SEED            = 0

# ⏱️ 예상 시간 안내 (이 단계가 가장 오래 걸립니다)
print(f"⏱️ 합성 시작 — 이미지 1장당 T4에서 약 20~40초.")
print(f"   IMGS_PER_PROMPT={IMGS_PER_PROMPT} 기준 보통 10~20분 소요"
      f" (첫 실행은 SDXL ~7GB 다운로드/로드 3~5분이 추가).")
print(f"   더 빨리 보려면 IMGS_PER_PROMPT를 낮추세요.\n")

with Timer("합성 결함 이미지 생성 (SDXL)"):
    subprocess.run([sys.executable, "generate_augmented_images.py",
                    "--src_dir", "/content/ksdd2_pre",
                    "--imgs_per_prompt", str(IMGS_PER_PROMPT),
                    "--seed", str(SEED)], check=True)

# 생성 결과 폴더(augmented_*)를 찾아 실제 생성된 장수를 센다
aug_dirs = sorted(glob.glob("/content/ksdd2_pre/**/augmented_*", recursive=True))
aug_imgs = []
for d in aug_dirs:
    aug_imgs += glob.glob(os.path.join(d, "*.png")) + glob.glob(os.path.join(d, "*.jpg"))
NUM_AUGMENTED = len(aug_imgs)
log.info(f"  생성된 합성 이미지: {NUM_AUGMENTED}장  (폴더: {aug_dirs})")

In [ ]:
# 생성된 합성 결함 이미지 몇 장 미리보기
with Timer("합성 결과 미리보기"):
    show = aug_imgs[:6]
    if show:
        fig, axes = plt.subplots(1, len(show), figsize=(3 * len(show), 3))
        if len(show) == 1:
            axes = [axes]
        for ax, p in zip(axes, show):
            ax.imshow(Image.open(p)); ax.axis("off")
            ax.set_title(os.path.basename(p), fontsize=8)
        plt.tight_layout(); plt.show()
    else:
        log.warning("  표시할 합성 이미지가 없습니다. 생성 셀을 먼저 확인하세요.")

### 5. 실험 — 합성 데이터가 탐지 성능(AP)을 올리는가

여기가 **VISION Track 2의 채점 기준**과 직접 닿는 부분입니다. 같은 ResNet-50을 두 가지 구성으로 학습해 **AP**를 비교합니다.

| 구성 | 실제 결함(GT) | 합성 결함 | 의미 |
|------|:---:|:---:|------|
| **Baseline** | ❌ (zero-shot) | ❌ | 합성 없이 — 기준선 |
| **DIAG 증강** | ❌ (zero-shot) | ✅ `NUM_AUGMENTED`장 | 합성으로 소수 클래스 보강 |

> **가장 중요한 원칙**
>
> <span style="color:#E74C3C; font-weight:bold">평가는 항상 실제 테스트셋으로</span> 합니다.
> 합성으로 평가하면 "합성을 잘 맞춘다"는 동어반복이 됩니다. DIAG의 평가 코드는 실제 KSDD2 테스트셋을 사용합니다.

In [ ]:
def run_training(tag, extra_args):
    """train_ResNet50.py를 실행하고 로그에서 AP를 파싱한다."""
    cmd = [sys.executable, "train_ResNet50.py",
           "--seed", str(SEED), "--epochs", "30", "--batch_size", "32",
           "--num_workers", "2", "--dataset_path", "/content/ksdd2_pre",
           "--zero_shot"] + extra_args
    log.info(f"[{tag}] 실행: {' '.join(cmd)}")
    out = subprocess.run(cmd, capture_output=True, text=True)
    full = out.stdout + "\n" + out.stderr
    print(full[-1500:])                     # 마지막 로그를 직접 확인
    # 'AP' 뒤에 오는 첫 번째 소수를 추출 (형식이 다르면 위 로그에서 직접 확인)
    m = re.findall(r"AP[^0-9]{0,6}([01]?\.\d+|\d{1,3}\.\d+)", full)
    ap = float(m[-1]) if m else None
    log.info(f"[{tag}] 파싱된 AP: {ap}")
    return ap

results = {}
with Timer("Baseline 학습 (합성 없음)"):
    results["Baseline"] = run_training("Baseline", [])

with Timer("DIAG 증강 학습 (합성 추가)"):
    results["DIAG 증강"] = run_training(
        "DIAG", ["--add_augmented", "--num_augmented", str(NUM_AUGMENTED)])

In [ ]:
# 결과 비교 — AP 막대그래프
with Timer("결과 비교"):
    labels = list(results.keys())
    vals = [results[k] if results[k] is not None else 0 for k in labels]
    colors = ["#95A5A6", "#27AE60"]

    plt.figure(figsize=(5, 4))
    bars = plt.bar(labels, vals, color=colors[:len(labels)])
    for b, v in zip(bars, vals):
        plt.text(b.get_x() + b.get_width()/2, v + 0.005, f"{v:.3f}",
                 ha="center", fontweight="bold")
    plt.ylabel("AP (Average Precision)")
    plt.title("합성 데이터 유무에 따른 탐지 성능")
    plt.ylim(0, 1.0)
    plt.tight_layout(); plt.show()

    if all(results[k] is not None for k in results):
        gain = results["DIAG 증강"] - results["Baseline"]
        log.info(f"  AP 변화: {results['Baseline']:.3f} → {results['DIAG 증강']:.3f} "
                 f"({gain:+.3f})")
    else:
        log.warning("  AP 파싱에 실패한 항목이 있습니다. 위 학습 로그에서 AP 값을 직접 확인하세요.")

### 마무리 — VISION Track 2 관점에서 정리

이번 실습에서 우리는 대회 과제를 그대로 따라가 봤습니다.

- **과제**: 제한된 라벨 + 합성으로 탐지 AP 올리기 (VISION Track 2)
- **방법**: DIAG — 텍스트 + 영역 지정으로 정상 이미지 위에 결함을 학습 없이 생성
- **결과**: 합성 결함을 더했을 때 ResNet-50의 AP가 어떻게 변하는지 직접 확인

> **기억할 점 — "잘" 생성해야 한다**
>
> 합성은 <span style="color:#E74C3C; font-weight:bold">적당히, 잘 만들 때</span>만 도움이 됩니다.
> 진짜 결함과 동떨어진 합성물은 오히려 탐지기를 망칩니다.
> 그래서 DIAG처럼 **in-distribution(원 분포 안)** 으로 생성하고, 효과는 **실제 테스트셋의 AP**로만 검증합니다.
> 이것이 VISION Track 2가 묻는 진짜 질문 — *"네가 만든 데이터가 정말 성능을 올리는가?"* — 입니다.

> **더 해보기**
>
> - `IMGS_PER_PROMPT`를 키워 합성량을 늘리고 AP 변화를 관찰
> - DIAG 논문은 양성 샘플이 있을 때 AP +18%, 없을 때 +28% 향상을 보고
> - 이미지-마스크 쌍 생성을 보고 싶다면 **AnomalyDiffusion**(`sjtuplayer/anomalydiffusion`, MVTec/VisA)
> - 원 출처: [DIAG (CBMI 2024)](https://intelligolabs.github.io/DIAG/) · [VISION Workshop](https://vision-based-industrial-inspection.github.io/)
